In [52]:
from sentence_transformers import SentenceTransformer
from typing import List, Dict, TypedDict, Any
import json 
from pathlib import Path 
from transformers import AutoTokenizer
import numpy as np

In [44]:
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")

In [45]:
def load_chunks(path: Path) -> Dict | List:
    with open(path, "r", encoding="utf-8") as f:
        content = json.load(f)
    return content

In [46]:
chunks_1a = load_chunks(Path("chunks_1a.json"))
chunks_1b = load_chunks(Path("chunks_1b.json"))

print(type(chunks_1a))
print(type(chunks_1b))

<class 'dict'>
<class 'list'>


In [35]:
def iter_chunks(chunks):
    if isinstance(chunks, dict):
        for chunk_list in chunks.values():
            for chunk in chunk_list:
                yield chunk
    elif isinstance(chunks, list):
        yield from chunks

In [36]:
def token_count(text):
    return len(tokenizer.encode(text, add_special_tokens=False))

In [47]:
def mark_truncated(chunks, limit=512):
    flagged = 0
    for chunk in iter_chunks(chunks):
        is_over = token_count(chunk["text"]) > limit
        chunk["truncated"] = is_over
        flagged += is_over
    return flagged

In [48]:
flagged_1a = mark_truncated(chunks_1a)
flagged_1b = mark_truncated(chunks_1b)
print(flagged_1a, flagged_1b)   # sanity check: should print 3 19

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (533 > 512). Running this sequence through the model will result in indexing errors


3 19


In [49]:
with open("chunks_1a_marked.json", "w", encoding="utf-8") as f:
    json.dump(chunks_1a, f, indent=2, ensure_ascii=False)

with open("chunks_1b_marked.json", "w", encoding="utf-8") as f:
    json.dump(chunks_1b, f, indent=2, ensure_ascii=False)

In [37]:
count_1a = sum(1 for chunk in iter_chunks(chunks_1a) if token_count(chunk["text"]) > 512)
count_1b = sum(1 for chunk in iter_chunks(chunks_1b) if token_count(chunk["text"]) > 512)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (533 > 512). Running this sequence through the model will result in indexing errors


In [38]:
print(f"Chunks 1a count: {count_1a}")
print(f"Chunks 1b count: {count_1b}")

Chunks 1a count: 3
Chunks 1b count: 19


### Final

In [54]:
def load_chunks(path: Path) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        content = json.load(f)
    if isinstance(content, dict):
        return [chunk for chunk_list in content.values() for chunk in chunk_list]
    return content

In [55]:
chunks_1a = load_chunks(Path("chunks_1a_marked.json"))
chunks_1b = load_chunks(Path("chunks_1b_marked.json"))

In [56]:
print(len(chunks_1a), len(chunks_1b))

126 182


In [51]:
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6383.58it/s]


In [57]:
def embed_corpus(chunks, model):
    texts = [chunk["text"] for chunk in chunks]
    embeddings = model.encode(
        texts, 
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    return embeddings

In [58]:
embed_1a = embed_corpus(chunks_1a, model)
embed_1b = embed_corpus(chunks_1b, model)

print(embed_1a.shape)
print(embed_1b.shape)

Batches: 100%|██████████| 6/6 [00:02<00:00,  2.62it/s]

(126, 384)
(182, 384)


In [59]:
def save_embeddings(embeddings, chunks, name, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    np.save(output_dir / f"{name}_embeddings.npy", embeddings)
    
    chunk_ids = [chunk["chunk_id"] for chunk in chunks]
    with open(output_dir / f"{name}_ids.json", "w", encoding="utf-8") as f:
        json.dump(chunk_ids, f, indent=2)

In [61]:
output_dir = Path("embeddings/")
save_embeddings(embed_1a, chunks_1a, "1a", output_dir=output_dir)
save_embeddings(embed_1b, chunks_1b, "1b", output_dir=output_dir)

In [62]:
def embed_query(query, model):
    prefixed = "Represent this sentence for searching passages" + query
    return model.encode([prefixed], normalize_embeddings=True)[0]

In [ ]:
query = "According to its docstring, since what version has the module-level session() function been deprecated?"

vec_with_prefix = embed_query(query, model)
vec_no_prefix = model.encode([query], normalize_embeddings=True)[0]

sims_with_prefix = embed_1a @ vec_with_prefix # @-> matrix vector multiplication
sims_no_prefix = embed_1a @ vec_no_prefix 

top_k_with_prefix = np.argsort(sims_with_prefix)[::-1][:3]
top_k_no_prefix = np.argsort(sims_no_prefix)[::-1][:3]

for i in top_k_with_prefix:
    print(chunks_1a[i]["chunk_id"], sims_with_prefix[i])
print("------------")
for i in top_k_no_prefix:
    print(chunks_1a[i]["chunk_id"], sims_no_prefix[i])